In [ ]:
import pandas as pd

In [ ]:
df_telco = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
print("Forme :", df_telco.shape)
print(df_telco.dtypes)
df_telco.head()

In [ ]:
def audit_qualite(df):
    print(f"Forme : {df.shape}")
    print("\nTypes des colonnes :")
    print(df.dtypes)

    print("\nPourcentage de valeurs manquantes :")

    manquants = (
        df.isna().mean().mul(100).sort_values(ascending=False)
    )
    nb_colonnes_manquantes = (manquants > 0).sum()
    print(
        f"Manquants détectés : "
        f"{nb_colonnes_manquantes} colonne(s)"
    )

    for colonne, pct in manquants.items():
        print(f"{colonne:<25} {pct:.2f}%")
    
    if "Churn" not in df.columns:
        print("\n⚠️ Colonne 'Churn' absente.")
        return

    print("\nRépartition de Churn :")

    counts = df["Churn"].value_counts().sort_index()
    total = len(df)

    for valeur, nb in counts.items():
        pct = 100 * nb / total if total > 0 else 0
        print(f"Churn {valeur} : {nb} ({pct:.1f}%)")

In [ ]:
print("=======Audit qualité sur le dataset complet=======")
audit_qualite(df_telco)
print("\n =======Audit qualité sur la classe 'yes' de churn=======")
audit_qualite(df_telco[df_telco["Churn"] == "Yes"])
print("\n =======Audit qualité sur la classe 'no' de churn=======")
audit_qualite(df_telco[df_telco["Churn"] == "No"])

In [ ]:
def reparer_total_charges(df):
    df_repare = df.copy()
    
    if "TotalCharges" not in df.columns:
        print("\n⚠️ Colonne 'TotalCharges' absente.")
        return
    
    audit_virgule = (
        df["MonthlyCharges"]
        .astype(str)
        .str.contains(",", regex=False)
    )

    print(
        "Valeurs contenant une virgule :",
        audit_virgule.sum()
    )

    total_charges_num = pd.to_numeric(
        df_repare["TotalCharges"],
        errors="coerce" 
    )

    nb_nan = total_charges_num.isna().sum()
    print(f"Trous démasqués : {nb_nan}")

    if nb_nan == len(df_repare):
        print("\n⚠️ Échec de conversion : la colonne est 100% non numérique.")
        return
    
    df_repare["TotalCharges"] = total_charges_num

    mediane = df_repare["TotalCharges"].median()
    df_repare["TotalCharges"] = (
        df_repare["TotalCharges"]
        .fillna(mediane)
    )
    print(
        f"Type final : {df_repare['TotalCharges'].dtype}"
    )

    return df_repare

In [ ]:
print("\n =======Réparer total charges=======")
df_telco = reparer_total_charges(df_telco)

In [ ]:
def encoder_features(df):
    df_encode = df.copy()
    if "customerID" in df_encode.columns:
        df_encode = df_encode.drop(columns=["customerID"])

    colonnes_obj = df_encode.select_dtypes(include=["object", "string"]).columns.tolist()

    # On ne touche pas à la cible si elle existe
    if "Churn" in colonnes_obj:
        colonnes_obj.remove("Churn")

    colonnes_binaires_y_n = []
    for col in colonnes_obj:
        valeurs = set(df_encode[col].dropna().unique())
        if valeurs <= {"Yes", "No"}:
            colonnes_binaires_y_n.append(col)
    
    for col in colonnes_binaires_y_n:
        df_encode[col] = df_encode[col].map({
            "No": 0,
            "Yes": 1
        })
    
    colonnes_nominales = [
        c for c in colonnes_obj
        if c not in colonnes_binaires_y_n
    ]

    # One-Hot Encoding
    df_encode = pd.get_dummies(
        df_encode,
        columns=colonnes_nominales,
        drop_first=False,
        dtype=int
    )

    if "Churn" in df_encode.columns:
        df_encode["Churn"] = df_encode["Churn"].map({
            "No": 0,
            "Yes": 1
        })
    
    print(f"Forme après encodage : {df_encode.shape}")

    return df_encode

In [ ]:
print("\n =======Encodage des catégorielles=======")
df_telco = encoder_features(df_telco)

In [ ]:
def detecter_outliers_iqr(df, colonne):
    if colonne not in df.columns:
        print(f"Colonne {colonne} absente.")
        return
    
    serie = df[colonne].dropna()

    if serie.empty:
        return None, None, 0
    
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)

    IQR = Q3 - Q1

    borne_basse = Q1 - 1.5 * IQR
    borne_haute = Q3 + 1.5 * IQR

    outliers = serie[(serie < borne_basse) | (serie > borne_haute)]

    nb_outliers = len(outliers)

    print(f"\nColonne : {colonne}")
    print(f"Borne basse : {borne_basse:.2f}")
    print(f"Borne haute : {borne_haute:.2f}")
    print(f"Outliers détectés : {nb_outliers}")

    return borne_basse, borne_haute, nb_outliers

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
def boxplot_colonne(df, colonne):
    plt.figure(figsize=(6, 6))
    plt.boxplot(df[colonne].dropna(), vert=False)
    plt.title(f"Boxplot - {colonne}")
    plt.xlabel(colonne)
    plt.show()

In [ ]:
print("\n =======Outliers=======")

colonnes_df_telco = ["tenure", "MonthlyCharges", "TotalCharges"]

for col in colonnes_df_telco:
    detecter_outliers_iqr(df_telco, col)
    boxplot_colonne(df_telco, col)